# Notebook 07 — Final production SMAP QA dataset (2019–2022)

This is the **full research run**, not a pilot. It reuses the original Notebook 07
SMAP target and predictor definitions. The Earth Engine project is
`axial-paratext-491911-b6`.

Run cells **1–4** to submit **only missing quarters**. Previously downloaded CSVs,
and Earth Engine tasks that are already queued, running or completed, are skipped.
The submission code runs immediately: do not repeatedly press Run All.

Earth Engine exports are asynchronous. When every job finishes, download the
CSV files from Google Drive folder `CHT_smap_QA_exports` into
`data/raw/07_smap_ee_exports/`, then run the final consolidation cell.
The final cell requires **all 16 quarters**, checks duplicate SMAP posting/date
keys and writes the standard Notebook 08 input. It cannot create missing
satellite data, and the QA target is not a ground measurement.


## 1. Initialize the full study and Earth Engine

In [1]:
from pathlib import Path
from shapely.geometry import mapping
import hashlib, json
import geopandas as gpd
import pandas as pd
import numpy as np
import ee

ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
             if (p / 'configs' / 'study_config.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Launch Jupyter from the project root or notebooks/.')
CFG = json.loads((ROOT / 'configs' / 'study_config.json').read_text(encoding='utf-8'))
assert CFG['gee_project_id'] == 'axial-paratext-491911-b6'
boundary = ROOT / CFG['boundary_relative_path']
if not boundary.exists():
    raise FileNotFoundError(f'Boundary shapefile missing: {boundary}')
districts = gpd.read_file(boundary)
assert districts.crs is not None, 'Boundary CRS missing; identify it before use.'
assert districts.geometry.notna().all() and districts.geometry.is_valid.all()
assert len(districts) == 3
assert set(districts[CFG['district_name_field']].astype(str)) == set(CFG['expected_districts'])
districts = districts.to_crs('EPSG:4326')

ee.Initialize(project=CFG['gee_project_id'])  # Must precede ALL ee.Geometry/Image constructors.
features = [ee.Feature(ee.Geometry(mapping(row.geometry.simplify(0.0005, preserve_topology=True))),
                       {'district': str(row[CFG['district_name_field']])})
            for _, row in districts.iterrows()]
DISTRICT_FC = ee.FeatureCollection(features)
AOI = DISTRICT_FC.geometry()

# Final production coverage: all 16 calendar quarters, 2019–2022.
FULL_YEARS = [2019, 2020, 2021, 2022]
DRIVE_FOLDER = 'CHT_smap_QA_exports'
RAW_DIR = ROOT / 'data' / 'raw' / '07_smap_ee_exports'
PROCESSED_DIR = ROOT / 'data' / 'processed'
TABLE_DIR = ROOT / 'outputs' / 'tables'

def full_quarters(years):
    return [(f'{year}_Q{q}', f'{year}-{first:02d}-01',
             f'{year + (q == 4)}-{(first + 3 - 1) % 12 + 1:02d}-01')
            for year in years for q, first in enumerate((1, 4, 7, 10), start=1)]

periods = full_quarters(FULL_YEARS)
EXPECTED_PERIODS = {name: (start, end) for name, start, end in periods}
print('Earth Engine cloud project:', CFG['gee_project_id'])
print('Study districts:', list(districts[CFG['district_name_field']]))
print('Complete study design:', len(periods), 'quarters, 2019–2022')


Earth Engine cloud project: axial-paratext-491911-b6
Study districts: ['Bandarban', 'Khagrachhari', 'Rangamati']
Complete study design: 16 quarters, 2019–2022


## 2. Load authoritative data sources and grid projection

In [2]:
SMAP = (ee.ImageCollection('NASA/SMAP/SPL3SMP_E/005')
        .select(['soil_moisture_am', 'retrieval_qual_flag_am']))
CHIRPS = ee.ImageCollection('UCSB-CHC/CHIRPS/V3/DAILY_RNL').select('precipitation')
S1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(AOI)
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .select(['VV', 'VH']))
MODIS = (ee.ImageCollection('MODIS/061/MOD13Q1')
         .filterBounds(AOI).select(['EVI', 'SummaryQA']))
ELEV = ee.Image('USGS/SRTMGL1_003').select('elevation').rename('elevation_m')
SLOPE = ee.Terrain.slope(ELEV).rename('slope_deg')

reference = ee.Image(SMAP.filterDate('2021-01-01', '2021-02-01').first())
GRID_PROJ = reference.select('soil_moisture_am').projection()
GRID_XY = ee.Image.pixelCoordinates(GRID_PROJ).rename(['grid_x', 'grid_y'])
MASKED_S1 = ee.Image.constant([0, 0]).rename(['s1_vv_db', 's1_vh_db']).updateMask(ee.Image.constant(0))
MASKED_EVI = ee.Image.constant(0).rename('evi_strict_qa').updateMask(ee.Image.constant(0))

print('SMAP reference projection:', GRID_PROJ.getInfo())
print('2021 SMAP images:', SMAP.filterDate('2021-01-01', '2022-01-01').size().getInfo())


SMAP reference projection: {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.08, 0, -179.99999994985163, 0, -0.08, 85.04450187956549]}
2021 SMAP images: 365


## 3. Build the established QA target and lagged predictors

In [3]:
def predictor_image(smap_image):
    sm = ee.Image(smap_image)
    day = ee.Date(sm.get('system:time_start'))
    rain_day = CHIRPS.filterDate(day, day.advance(1, 'day')).sum().rename('rain_day_mm')
    rain_prior1 = CHIRPS.filterDate(day.advance(-1, 'day'), day).sum().rename('rain_prior1_mm')
    rain_prior7 = CHIRPS.filterDate(day.advance(-7, 'day'), day).sum().rename('rain_prior7_mm')

    radar_collection = S1.filterDate(day.advance(-24, 'day'), day)
    radar_count = radar_collection.size()
    radar = ee.Image(ee.Algorithms.If(
        radar_count.gt(0), radar_collection.median().rename(['s1_vv_db', 's1_vh_db']),
        MASKED_S1))
    radar_difference = radar.select('s1_vh_db').subtract(radar.select('s1_vv_db')).rename('s1_vh_minus_vv_db')

    # MOD13Q1 is a 16-day composite; avoid a nominal start date that could
    # include imagery acquired after this SMAP AM retrieval.
    evi_collection = MODIS.filterDate(day.advance(-48, 'day'), day.advance(-15, 'day'))
    evi_count = evi_collection.size()
    evi = ee.Image(ee.Algorithms.If(
        evi_count.gt(0),
        ee.Image(evi_collection.sort('system:time_start', False).first())
          .select('EVI').multiply(0.0001).rename('evi_strict_qa')
          .updateMask(ee.Image(evi_collection.sort('system:time_start', False).first())
                      .select('SummaryQA').eq(0)),
        MASKED_EVI))

    return ee.Image.cat([
        GRID_XY, sm.select('soil_moisture_am').rename('smap_sm_m3m3'),
        sm.select('retrieval_qual_flag_am'), rain_day, rain_prior1, rain_prior7,
        radar, radar_difference, evi, ELEV, SLOPE,
        ee.Image.constant(radar_count).rename('s1_scene_count_prior24d'),
        ee.Image.constant(evi_count).rename('evi_scene_count_prior48to16d')
    ]).toFloat().unmask(-9999)

EXPORT_FIELDS = [
    'district', 'date', 'grid_x', 'grid_y', 'smap_sm_m3m3',
    'retrieval_qual_flag_am', 'rain_day_mm', 'rain_prior1_mm', 'rain_prior7_mm',
    's1_vv_db', 's1_vh_db', 's1_vh_minus_vv_db',
    'evi_strict_qa', 'elevation_m', 'slope_deg',
    's1_scene_count_prior24d', 'evi_scene_count_prior48to16d'
]

def one_day_features(smap_image, region_feature):
    sm = ee.Image(smap_image)
    district = ee.Feature(region_feature)
    day_text = ee.Date(sm.get('system:time_start')).format('YYYY-MM-dd')
    return predictor_image(sm).sample(
        region=district.geometry(), projection=GRID_PROJ,
        dropNulls=False, geometries=False, tileScale=4
    ).map(lambda f: ee.Feature(f).set({'district': district.get('district'),
                                      'date': day_text}))

def export_table_for_period(start, end):
    images = SMAP.filterBounds(AOI).filterDate(start, end).sort('system:time_start')
    length = images.size()
    days = images.toList(length)

    def samples_for_image(i):
        image = ee.Image(days.get(i))
        all_districts = DISTRICT_FC.toList(DISTRICT_FC.size())
        district_lists = ee.List.sequence(0, DISTRICT_FC.size().subtract(1)).map(
            lambda j: one_day_features(image, ee.Feature(all_districts.get(j))))
        return ee.FeatureCollection(district_lists).flatten()

    all_dates = ee.List.sequence(0, length.subtract(1)).map(samples_for_image)
    return ee.FeatureCollection(all_dates).flatten()


## 4. Submit all missing quarterly exports now

In [4]:
from datetime import datetime, timezone
import csv
import re

# PRODUCTION: this cell submits missing quarters immediately. No pilot mode.
# Before re-running, check the Earth Engine Tasks page and this cell's receipt.
# Existing local quarter CSVs and Earth Engine queued/running/completed tasks
# are recognized so that previously finished quarters are not re-submitted.
NAME_RE = re.compile(r"^07_cht_smap_(20(?:19|20|21|22)_Q[1-4])(?:_pilot)?\.csv$")
TASK_RE = re.compile(r"^07_cht_smap_(20(?:19|20|21|22)_Q[1-4])(?:_pilot)?$")

RAW_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

local_quarters = {}
for path in sorted(RAW_DIR.glob('07_cht_smap_*.csv')):
    match = NAME_RE.fullmatch(path.name)
    if not match:
        raise ValueError(f'Unexpected raw export file: {path.name}. Move it out until resolved.')
    quarter = match.group(1)
    if quarter in local_quarters:
        raise ValueError(f'Two raw CSVs for {quarter}: {local_quarters[quarter].name} and {path.name}')
    if path.stat().st_size == 0:
        raise ValueError(f'Empty raw CSV file: {path.name}')
    with path.open(encoding='utf-8-sig', newline='') as fh:
        headers = set(next(csv.reader(fh)))
    if missing_fields := (set(EXPORT_FIELDS) - headers):
        raise ValueError(f'Incomplete columns in {path.name}: {sorted(missing_fields)}')
    local_quarters[quarter] = path

remote_tasks = {}
for task in ee.batch.Task.list():
    description = (task.config or {}).get('description', '')
    match = TASK_RE.fullmatch(description)
    if match:
        quarter = match.group(1)
        state = getattr(task.state, 'value', str(task.state))
        remote_tasks.setdefault(quarter, []).append({
            'state': state, 'id': task.id, 'description': description})

receipt = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'gee_project': CFG['gee_project_id'],
    'years': FULL_YEARS,
    'product': 'NASA/SMAP/SPL3SMP_E/005',
    'target': 'QA accepted enhanced SMAP AM; not in-situ ground observations',
    'already_downloaded': [], 'already_in_earth_engine': [],
    'submitted_now': [], 'submission_errors': [],
}
receipt_path = TABLE_DIR / '07_full_submission_receipt.json'

def save_receipt():
    receipt_path.write_text(json.dumps(receipt, indent=2), encoding='utf-8')

save_receipt()
for name, start, end in periods:
    if name in local_quarters:
        print('LOCAL:', name, local_quarters[name].name)
        receipt['already_downloaded'].append(name)
        save_receipt()
        continue

    already_submitted = [
        record for record in remote_tasks.get(name, [])
        if record['state'] in {'READY', 'RUNNING', 'COMPLETED', 'CANCEL_REQUESTED'}
    ]
    if already_submitted:
        best = already_submitted[0]
        print('EXISTING TASK:', name, best['state'], best['id'],
              '(download the CSV after completion)')
        receipt['already_in_earth_engine'].append({'quarter': name, **best})
        save_receipt()
        continue

    try:
        n_images = SMAP.filterBounds(AOI).filterDate(start, end).size().getInfo()
        if n_images <= 0:
            raise ValueError('No SMAP images found; export cannot proceed')
        task = ee.batch.Export.table.toDrive(
            collection=export_table_for_period(start, end),
            description=f'07_cht_smap_{name}',
            folder=DRIVE_FOLDER,
            fileNamePrefix=f'07_cht_smap_{name}',
            fileFormat='CSV', selectors=EXPORT_FIELDS,
        )
        task.start()
        entry = {'quarter': name, 'task_id': task.id, 'SMAP_images': int(n_images)}
        receipt['submitted_now'].append(entry)
        print('SUBMITTED:', name, '| SMAP images:', n_images, '| task:', task.id)
    except Exception as exc:
        print('NOT SUBMITTED:', name, type(exc).__name__, str(exc))
        receipt['submission_errors'].append({'quarter': name, 'error': str(exc)})
    finally:
        # Save after *each* attempt, including a partial submission run.
        save_receipt()

print('\nSubmission summary')
for key in ('already_downloaded', 'already_in_earth_engine',
            'submitted_now', 'submission_errors'):
    print(f'  {key}:', len(receipt[key]))
print('Receipt:', receipt_path)
print('Earth Engine tasks: https://code.earthengine.google.com/tasks')
print('Download completed CSVs to:', RAW_DIR)


LOCAL: 2019_Q1 07_cht_smap_2019_Q1.csv
LOCAL: 2019_Q2 07_cht_smap_2019_Q2.csv
SUBMITTED: 2019_Q3 | SMAP images: 70 | task: WHLOIKY7KXK7K537ZWZ6YSQ4
SUBMITTED: 2019_Q4 | SMAP images: 92 | task: DTPQLPLX5MKWHPNET6A5FOQ6
LOCAL: 2020_Q1 07_cht_smap_2020_Q1.csv
LOCAL: 2020_Q2 07_cht_smap_2020_Q2.csv
SUBMITTED: 2020_Q3 | SMAP images: 92 | task: V5LO334LW4AES3XV6QQSZLBJ
SUBMITTED: 2020_Q4 | SMAP images: 92 | task: X7XXMLZYT74OYAHU3PTP4MFM
LOCAL: 2021_Q1 07_cht_smap_2021_Q1_pilot.csv
LOCAL: 2021_Q2 07_cht_smap_2021_Q2.csv
LOCAL: 2021_Q3 07_cht_smap_2021_Q3.csv
SUBMITTED: 2021_Q4 | SMAP images: 92 | task: UMRLIKFIOE3KI5CCAW2QOLEJ
SUBMITTED: 2022_Q1 | SMAP images: 90 | task: TMQ2N42SKLMDHAFJAMOKPZNL
SUBMITTED: 2022_Q2 | SMAP images: 91 | task: H2BLKSOHSOFIKX37VEDGGLWN
SUBMITTED: 2022_Q3 | SMAP images: 45 | task: HPXPV4XMHYNK2BAIPXUCUC3E
SUBMITTED: 2022_Q4 | SMAP images: 91 | task: DTRAXYH24EEGT3PN23Z3QE3W

Submission summary
  already_downloaded: 7
  already_in_earth_engine: 0
  submitted_now: 9

## 5. Check the Earth Engine queue (optional)

You may rerun the next cell without submitting more work. Do not download a
quarter until its Earth Engine task is `COMPLETED`. The preceding cell's
receipt records submissions even if a later task produces an error.


In [7]:
# Optional status refresh; does NOT submit new exports.
states = {}
for task in ee.batch.Task.list():
    description = (task.config or {}).get('description', '')
    match = TASK_RE.fullmatch(description)
    if match and match.group(1) in EXPECTED_PERIODS:
        state = getattr(task.state, 'value', str(task.state))
        states.setdefault(match.group(1), []).append(state)
for name, _, _ in periods:
    if name in local_quarters:
        print(name, 'CSV PRESENT')
    else:
        print(name, ', '.join(states.get(name, ['NO VISIBLE TASK / CSV'])))


2019_Q1 CSV PRESENT
2019_Q2 CSV PRESENT
2019_Q3 COMPLETED
2019_Q4 COMPLETED
2020_Q1 CSV PRESENT
2020_Q2 CSV PRESENT
2020_Q3 COMPLETED
2020_Q4 COMPLETED
2021_Q1 CSV PRESENT
2021_Q2 CSV PRESENT
2021_Q3 CSV PRESENT
2021_Q4 COMPLETED
2022_Q1 COMPLETED
2022_Q2 COMPLETED
2022_Q3 COMPLETED
2022_Q4 COMPLETED


## 6. Finalize the complete 2019–2022 dataset

**Wait until all 16 CSVs are on your PC.** Then run this cell. A source quarter
can contain fewer available SMAP dates than calendar days; the date-audit CSV
documents that limitation rather than manufacturing observations.

If it reports missing quarters, download those files and rerun **this cell only**.
After restarting your Jupyter kernel, rerun Sections 1–3 and this final
section. You do not have to resubmit the export section.
Notebook 08 can start only after this cell prints `COMPLETE` and you inspect the
QA coverage and date audit.


In [8]:
# Run AFTER downloading all 16 exports. No simulated targets, no silent
# substitution: an incomplete year blocks the final model-ready dataset.
import hashlib
import re

# Defined here as well so that, after reopening Jupyter, you can execute
# setup/data/predictor cells and this cell without resubmitting exports.
NAME_RE = re.compile(r"^07_cht_smap_(20(?:19|20|21|22)_Q[1-4])(?:_pilot)?\.csv$")

files_by_quarter = {}
for file in sorted(RAW_DIR.glob('07_cht_smap_*.csv')):
    match = NAME_RE.fullmatch(file.name)
    if not match:
        raise ValueError(f'Unexpected file name: {file.name}')
    quarter = match.group(1)
    if quarter in files_by_quarter:
        raise ValueError(f'Duplicate CSV for {quarter}: {file.name}')
    files_by_quarter[quarter] = file

missing = [name for name in EXPECTED_PERIODS if name not in files_by_quarter]
if missing:
    raise FileNotFoundError(
        f'Full 2019–2022 dataset is NOT ready. Missing {len(missing)} quarters: '
        + ', '.join(missing) + '. Download the exports from Google Drive '
        + f'into {RAW_DIR}, then rerun this cell.')

frames = []
quarter_audit = []
for name, start, end in periods:
    file = files_by_quarter[name]
    frame = pd.read_csv(file)
    lacking = set(EXPORT_FIELDS) - set(frame.columns)
    if lacking:
        raise ValueError(f'{file.name}: missing columns {sorted(lacking)}')
    frame = frame[EXPORT_FIELDS].copy()
    frame['date'] = pd.to_datetime(frame['date'], format='%Y-%m-%d', errors='raise')
    if frame.empty:
        raise ValueError(f'{name}: empty export')
    if not frame.date.between(pd.Timestamp(start),
                              pd.Timestamp(end) - pd.Timedelta(days=1)).all():
        raise ValueError(f'{name}: dates outside quarterly boundaries')
    expected_dates = pd.date_range(start, end, inclusive='left')
    actual_dates = pd.DatetimeIndex(frame.date.unique())
    quarter_audit.append({
        'quarter': name, 'rows': len(frame),
        'expected_calendar_days': len(expected_dates),
        'exported_product_dates': len(actual_dates),
        'missing_calendar_dates': ','.join(
            d.strftime('%Y-%m-%d') for d in expected_dates.difference(actual_dates)),
    })
    frames.append(frame)
    print('READ:', name, len(frame), 'rows,', len(actual_dates), 'product dates')

table = pd.concat(frames, ignore_index=True)
for col in (c for c in EXPORT_FIELDS if c not in {'district', 'date'}):
    table[col] = pd.to_numeric(table[col], errors='coerce')
    table.loc[table[col] <= -9998, col] = np.nan

assert set(table['district'].unique()) == set(CFG['expected_districts'])
if table[['date', 'district', 'grid_x', 'grid_y']].isna().any().any():
    raise ValueError('Missing date/district/grid coordinates in source exports')

table['grid_x'] = table['grid_x'].astype('int64')
table['grid_y'] = table['grid_y'].astype('int64')
table['grid_id'] = table['grid_x'].astype(str) + '_' + table['grid_y'].astype(str)
key = ['date', 'grid_id']
if table.duplicated(key).any():
    examples = table.loc[table.duplicated(key, keep=False),
                         ['date', 'grid_id', 'district']].head(12)
    raise ValueError(f'Duplicate SMAP posting/date keys:\n{examples}')

table['target_qa_valid'] = (
    table['retrieval_qual_flag_am'].eq(0)
    & table['smap_sm_m3m3'].between(0, 1, inclusive='both')
)
table.loc[~table.target_qa_valid, 'smap_sm_m3m3'] = np.nan
table = table.sort_values(['date', 'district', 'grid_x', 'grid_y']).reset_index(drop=True)
supported = table[table.target_qa_valid]
summary = (table.groupby('district', dropna=False)
           .agg(rows=('target_qa_valid', 'size'),
                qa_accepted=('target_qa_valid', 'sum'),
                distinct_dates=('date', 'nunique'),
                distinct_grids=('grid_id', 'nunique'))
           .reset_index())

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
destination = PROCESSED_DIR / '07_smap_ml_table.csv'
table.to_csv(destination, index=False, date_format='%Y-%m-%d')
summary.to_csv(TABLE_DIR / '07_smap_qa_support_by_district.csv', index=False)
pd.DataFrame(quarter_audit).to_csv(
    TABLE_DIR / '07_final_quarter_date_audit.csv', index=False)
manifest = {
    'source_csvs': [{'name': f.name, 'sha256': hashlib.sha256(f.read_bytes()).hexdigest()}
                    for f in files_by_quarter.values()],
    'source_quarters': sorted(files_by_quarter),
    'n_source_quarters': len(files_by_quarter),
    'target': 'QA accepted enhanced SMAP AM, not in-situ soil moisture',
    'date_min': table.date.min().date().isoformat(),
    'date_max': table.date.max().date().isoformat(),
    'rows': int(len(table)),
    'accepted_rows': int(len(supported)),
    'accepted_distinct_dates': int(supported.date.nunique()),
    'accepted_distinct_grids': int(supported.grid_id.nunique()),
}
(TABLE_DIR / '07_training_data_manifest.json').write_text(
    json.dumps(manifest, indent=2), encoding='utf-8')
print('\nCOMPLETE: all 16 quarters accounted for.')
print('Records:', len(table), '| QA-accepted:', len(supported))
print('Accepted dates:', supported.date.nunique(),
      '| accepted SMAP grid postings:', supported.grid_id.nunique())
print('Data:', destination)
print('QA summary:', TABLE_DIR / '07_smap_qa_support_by_district.csv')
print('Manifest:', TABLE_DIR / '07_training_data_manifest.json')
print('Calendar-day product coverage:', TABLE_DIR / '07_final_quarter_date_audit.csv')


READ: 2019_Q1 16200 rows, 90 product dates
READ: 2019_Q2 14400 rows, 80 product dates
READ: 2019_Q3 12600 rows, 70 product dates
READ: 2019_Q4 16560 rows, 92 product dates
READ: 2020_Q1 16380 rows, 91 product dates
READ: 2020_Q2 16380 rows, 91 product dates
READ: 2020_Q3 16560 rows, 92 product dates
READ: 2020_Q4 16560 rows, 92 product dates
READ: 2021_Q1 16200 rows, 90 product dates
READ: 2021_Q2 16380 rows, 91 product dates
READ: 2021_Q3 16560 rows, 92 product dates
READ: 2021_Q4 16560 rows, 92 product dates
READ: 2022_Q1 16200 rows, 90 product dates
READ: 2022_Q2 16380 rows, 91 product dates
READ: 2022_Q3 8100 rows, 45 product dates
READ: 2022_Q4 16380 rows, 91 product dates

COMPLETE: all 16 quarters accounted for.
Records: 248400 | QA-accepted: 17653
Accepted dates: 663 | accepted SMAP grid postings: 39
Data: D:\Gihub_Projects\cht-forest-ecohydrology-ml-starter\cht-forest-ecohydrology-ml\data\processed\07_smap_ml_table.csv
QA summary: D:\Gihub_Projects\cht-forest-ecohydrology-ml-s